# Chapter 7 &mdash; $Eclosure$: What $\varepsilon$ Edges Do to Simulation

**Concept 6 of the Chapter 7 decomposition:** *$Eclosure$: What $\varepsilon$ Edges Do to Simulation*

Tokens "ooze" along $\varepsilon$ edges one way; $Eclosure(q)$ is everything reachable from $q$ for free.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter7/Concept-Eclosure/Concept-Eclosure.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.AnimateNFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


With $\varepsilon$ edges present, a token does not sit still: it **oozes** along every
$\varepsilon$ edge it can, *in the direction of the arrow only*.

$$Eclosure(S) = \{q : q \text{ reachable from some } s\in S \text{ by } \varepsilon\text{ edges alone}\}$$

Two properties matter:

* $S \subseteq Eclosure(S)$ &mdash; zero $\varepsilon$ steps is allowed;
* $Eclosure$ is **idempotent**: $Eclosure(Eclosure(S)) = Eclosure(S)$ &mdash; it is a
  least fixed point, computed by iterating until nothing is added.

It is also why $\varepsilon$ **cycles** are harmless.

## 2. Definitions

### A machine with an $\varepsilon$ chain and an $\varepsilon$ cycle

In [ ]:
N = md2mc('''NFA
I : '' -> A
A : '' -> B
B : '' -> A          !! an epsilon CYCLE
B : 0 -> F
A : 1 -> F
''')

### $Eclosure$, computed as a least fixed point

In [ ]:
def eclose(N, S):
    cur = set(S)
    while True:
        nxt = cur | {t for q in cur for t in step_nfa(N, q, '')}
        if nxt == cur: return cur
        cur = nxt

## 3. Tests

Our fixed point agrees with Jove's `Eclosure`.

In [ ]:
for S in [{'I'}, {'A'}, {'B'}, {'F'}, {'I','F'}]:
    mine, jove = eclose(N, S), Eclosure(N, S)
    print("Eclosure(%-9s) = %-18s  agrees: %s"
          % (sorted(S), sorted(jove), mine == jove))
    assert mine == jove

**Reflexive:** $S \subseteq Eclosure(S)$, always.

In [ ]:
for S in [{'I'}, {'B'}, {'F'}]:
    assert S <= Eclosure(N, S)
print("every state is in its own Eclosure -- zero epsilon steps counts")

**Idempotent:** closing twice adds nothing. That is what 'closure' means.

In [ ]:
for S in [{'I'}, {'A'}, {'B'}, {'I','F'}]:
    assert Eclosure(N, Eclosure(N, S)) == Eclosure(N, S)
print("Eclosure(Eclosure(S)) == Eclosure(S) for every set tried")

**Directional:** the arrow matters &mdash; `A` reaches `B`, but the reverse needs its own edge.

In [ ]:
one_way = md2mc('''NFA
I : '' -> A
A : 0 -> F
''')
print("Eclosure({I}) =", sorted(Eclosure(one_way, {'I'})))
print("Eclosure({A}) =", sorted(Eclosure(one_way, {'A'})), " <- does NOT contain I")
assert 'I' not in Eclosure(one_way, {'A'})

The $\varepsilon$ **cycle** terminates because the closure is a least fixed point.

In [ ]:
print("Eclosure({A}) in the cyclic machine :", sorted(Eclosure(N, {'A'})))
print("the loop A -> B -> A adds nothing new on the second pass, so it stops.")
assert Eclosure(N, {'A'}) == Eclosure(N, {'B'})

## 4. Animation

The $\varepsilon$ edges, drawn: tokens spread along them before any symbol is read.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateNFA import *
AnimateNFA(N, FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Compute $Eclosure$ by hand for a chain of five $\varepsilon$ edges.
2. Why must $Eclosure$ be applied *before* the first symbol as well as after each one?
3. What is $Eclosure(\emptyset)$?

In [ ]:
# Your work for the exercises above.